# MSc Final Project: Visible In-Progress (VIP) notebook

| Field | Details |
| :--- | :--- |
| **Project title** | Multimodal Brain Tumour Classification Using MRI and CT Images |
| **Student name / number** | Mahaboob Shaik, 24164385 |
| **Supervisor** | Wajih Islam |
| **GitHub repository** | https://github.com/shaikjr/msc_final_project_24164385 |
| **Overleaf project** | [Add Overleaf link once confirmed] |
| **Ethics approval** | Not required (publicly available anonymised medical imaging dataset) / [Update if Ethics ID applies] |

This notebook is your practical work: data, artefact, experiments and results. Your report (IPR 10%, FPR 70%) is written in the Overleaf template. Your demo/viva (20%) is run from this notebook.

### VIP Workflow Rules
1. Add your supervisor as a collaborator on your GitHub repository in week 1 (`Settings -> Collaborators -> Add people`).
2. Commit and push after every task, after every change, and whenever you change an existing solution:
   ```bash
   git status
   git add .
   git commit -m "CODE: what you did"
   git push


1. **Add your supervisor as a collaborator** on your GitHub repository in week 1 (Settings → Collaborators → Add people). Your supervisor cannot review your work without this access.
2. **Commit and push after every task, after every change, and whenever you change an existing solution.** Your history shows how your project developed.
```bash
git status
git add .
git commit -m "CODE: what you did"
git push
```
3. Start each commit message with the part you worked on: `DATA:` · `CODE:` · `EXP:` · `TEST:` · `RESULT:` · `FIX:`
   e.g. `FIX: changed learning rate from 0.01 to 0.001, accuracy up to 0.87`
4. **Develop at least two parts of your artefact on their own branches** (for example your baseline and your proposed method). See *Working on a branch* below.
5. **Before each supervision meeting**, push your latest work. Your supervisor reviews what changed.
6. Never commit personal data or data that needs ethics approval. Keep the repository private, and do not rewrite its history.

## Working on a branch

A branch lets you build one part of your project without breaking the working version on `main`. Develop **at least two parts** this way, e.g. `baseline` and `proposed-method`.

**1. Create a branch and work on it**
```bash
git checkout -b baseline
```
Commit as usual while you work:
```bash
git status
git add .
git commit -m "CODE: baseline model, first version"
git push -u origin baseline
```
(`-u origin baseline` is needed only the first time you push a new branch.)

**2. When the part works, merge it into `main`**
```bash
git checkout main
git merge baseline
git push
```

**3. Start the next part on a new branch**
```bash
git checkout -b proposed-method
```

Check which branch you are on with `git branch` (the current one has a `*`). On GitHub, your supervisor can see every branch and its commits.

# Imports

In [1]:
from pathlib import Path
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Reproducibility seed
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Project directory structure
ROOT_DIR = Path(".")
DATA_DIR = ROOT_DIR / "data"
FIG_DIR = ROOT_DIR / "figures"
RESULTS_DIR = ROOT_DIR / "results"

for folder in [DATA_DIR, FIG_DIR, RESULTS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Environment and project directories ready.")

Environment and project directories ready.


Save your work, then run these commands in the terminal:
```bash
git status
git add .
git commit -m "CODE: project set-up, folders and environment"
git push
```

# Research question and objectives

**Problem:**  
Single-modality neuroimaging (MRI alone or CT alone) can miss complementary diagnostic cues—MRI provides superior soft-tissue contrast while CT captures calcification and bone structures accurately. Automated brain tumour classification models often rely on a single modality or lack explainability for clinical decision support.

**Main research question:**  
To what extent does conditional dual-branch feature-level fusion of MRI and CT scans improve brain tumour classification accuracy, generalisation, and interpretability compared to single-modality baseline models?

**Sub-questions:**
1. How do standard single-modality CNN architectures (Basic CNN, ResNet18, EfficientNet-B0) perform on MRI and CT brain tumour datasets independently?
2. Does feature-level fusion of MRI and CT representations outperform single-modality baselines across precision, recall, F1-score, and ROC-AUC?
3. How reliably does Grad-CAM highlight tumour-relevant anatomical regions across fused versus single-modality predictions?

**Objectives (specific and measurable):**
1. Curate, clean, and standardise paired/aligned MRI and CT brain tumour image datasets with documented preprocessing pipelines and zero data leakage across train, validation, and test splits.
2. Implement and benchmark single-modality baseline models (Basic CNN, ResNet18, EfficientNet-B0) on a dedicated `baseline` branch.
3. Design and implement a conditional dual-branch feature-level fusion architecture with Grad-CAM explainability on a dedicated `proposed-method` branch.
4. Evaluate all models using stratified validation, held-out test metrics (Accuracy, Macro-F1, ROC-AUC), and unit tests, exporting publication-ready figures and LaTeX tables for the FPR.

# Data
*FPR: Methodology*

* **Source and licence:** [Insert dataset name, URL/source, and licence details here]
* **Ethics / GDPR:** The dataset contains fully anonymised, publicly available medical imaging scans with all patient-identifying metadata removed. Raw data is stored locally in `data/` (excluded from Git via `.gitignore`) and never pushed to GitHub.

### Data-quality check and cleaning log

| Step | Problem found | What I changed | Why |
| :--- | :--- | :--- | :--- |
| 1 | Varying image resolutions and colour channels across MRI and CT scans | Resized all images to `224x224` and standardised to 3-channel RGB tensors | Ensures uniform input dimensions for pretrained backbones (ResNet18 / EfficientNet-B0) |
| 2 | Corrupt or duplicate image files | Checked file integrity and removed exact image hashes | Prevents training errors and data leakage between splits |
| 3 | Unnormalised pixel intensity ranges | Applied channel-wise normalisation (`ImageNet` mean/std or `[0, 1]` rescaling) | Stabilises gradient updates and speeds up convergence |

In [2]:
# Load dataset metadata / file paths and verify class distribution
# Example structure (replace with your dataset loader):
# df = pd.read_csv(DATA_DIR / "metadata.csv")
# print(df.shape)
# display(df.head())
# display(df.describe(include="all"))

def clean_metadata(df: pd.DataFrame) -> pd.DataFrame:
    """Remove missing paths and duplicate records."""
    df_clean = df.dropna().drop_duplicates().reset_index(drop=True)
    return df_clean

print("Data loading and cleaning template ready.")

Data loading and cleaning template ready.


### Data-quality check and cleaning log

| Step | Problem found | What I changed | Why |
|---|---|---|---|
| 1 | | | |

In [3]:
import hashlib
from pathlib import Path
import pandas as pd
import numpy as np
from PIL import Image, UnidentifiedImageError

# 1. Scan raw image directories (expects data/raw/<modality>/<class_name>/*.jpg|png)
RAW_DATA_DIR = DATA_DIR / "raw"
CLEAN_DATA_DIR = DATA_DIR / "cleaned"
CLEAN_DATA_DIR.mkdir(parents=True, exist_ok=True)

TARGET_SIZE = (224, 224)
VALID_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}

records = []
seen_hashes = set()
corrupt_count = 0
duplicate_count = 0

for img_path in RAW_DATA_DIR.rglob("*"):
    if img_path.suffix.lower() not in VALID_EXTS:
        continue

    # Step 1: Check file integrity (skip corrupt/unreadable scans)
    try:
        with Image.open(img_path) as img:
            img.verify()
    except (UnidentifiedImageError, OSError):
        corrupt_count += 1
        continue

    # Step 2: Check for exact duplicate scans using MD5 hash
    file_hash = hashlib.md5(img_path.read_bytes()).hexdigest()
    if file_hash in seen_hashes:
        duplicate_count += 1
        continue
    seen_hashes.add(file_hash)

    # Step 3: Standardise mode to 3-channel RGB and resize to 224x224
    modality = img_path.parents[1].name if len(img_path.parents) > 2 else "unknown"
    label = img_path.parent.name

    save_dir = CLEAN_DATA_DIR / modality / label
    save_dir.mkdir(parents=True, exist_ok=True)
    clean_path = save_dir / f"{img_path.stem}.png"

    with Image.open(img_path) as img:
        img_rgb = img.convert("RGB").resize(TARGET_SIZE, Image.Resampling.BILINEAR)
        img_rgb.save(clean_path)

    records.append({
        "modality": modality,
        "label": label,
        "clean_path": str(clean_path),
        "md5_hash": file_hash,
    })

# Save cleaned manifest
df_clean = pd.DataFrame(records)
df_clean.to_csv(DATA_DIR / "cleaned_metadata.csv", index=False)

print(f"Cleaning complete: {len(df_clean)} valid scans saved.")
print(f"Removed {corrupt_count} corrupt files and {duplicate_count} duplicate files.")
if not df_clean.empty:
    display(df_clean.groupby(["modality", "label"]).size().unstack(fill_value=0))

Cleaning complete: 0 valid scans saved.
Removed 0 corrupt files and 0 duplicate files.


Save your work, then run these commands in the terminal:
```bash
git status
git add .
git commit -m "DATA: cleaned dataset, see cleaning log"
git push
```

# Exploratory analysis
*FPR: Methodology; Quality and Results*

In [4]:
# Generate and save high-resolution EDA figures for Overleaf (300 DPI)
# Example:
# fig, ax = plt.subplots(figsize=(7, 4))
# df["label"].value_counts().plot(kind="bar", ax=ax, color="#2b5c8f")
# ax.set_title("Class Distribution Across Brain Tumour Categories")
# ax.set_xlabel("Tumour Class")
# ax.set_ylabel("Number of Scans")
# fig.savefig(FIG_DIR / "eda_class_balance.png", dpi=300, bbox_inches="tight")
# plt.show()

print("EDA figure export template ready.")

EDA figure export template ready.


# Method and artefact
*FPR: Methodology*

**Why this method?**  
- Compare single-modality CNN backbones against dual-branch feature-level fusion using findings from `Literature_Extraction.xlsx`.

### Baseline
Built on branch `baseline` (`git checkout -b baseline`): single-modality transfer learning baselines (Basic CNN, ResNet18, EfficientNet-B0).

Save your work, then run these commands in the terminal:
```bash
git status
git add .
git commit -m "CODE: exploratory analysis and first figures"
git push
```

# Method and artefact
*FPR: Methodology*

**Why this method?** Compare with alternatives from the literature (use your extraction sheet).

### Baseline
A simple method to compare against. Build it on its own branch: `git checkout -b baseline`

In [5]:
# Build baseline model on branch: baseline
def build_baseline_model(num_classes: int = 4, lr: float = 1e-3):
    """Define and return the baseline single-modality model."""
    pass

print("Baseline module placeholder ready.")

Baseline module placeholder ready.


### Proposed method / artefact
Built on branch `proposed-method` (`git checkout -b proposed-method`): conditional dual-branch feature-level fusion combining MRI and CT representations with Grad-CAM visual explanations.

In [6]:
# Build proposed dual-branch fusion model on branch: proposed-method
def build_fusion_model(num_classes: int = 4, fusion_dim: int = 256, lr: float = 1e-4):
    """Define and return the conditional dual-branch MRI/CT fusion model."""
    pass

print("Proposed fusion module placeholder ready.")

Proposed fusion module placeholder ready.


Save your work, then run these commands in the terminal:
```bash
git status
git add .
git commit -m "CODE: baseline and proposed method"
git push
```

# Experiments
*FPR: Quality and Results*


In [7]:
# Track all experiment runs systematically and save to results/experiments.csv
experiments_log = [
    # {"exp_id": "E01", "branch": "baseline", "model": "Basic CNN (MRI)", "lr": 1e-3, "epochs": 20, "val_acc": 0.00, "macro_f1": 0.00},
    # {"exp_id": "E02", "branch": "baseline", "model": "ResNet18 (MRI)", "lr": 1e-4, "epochs": 20, "val_acc": 0.00, "macro_f1": 0.00},
    # {"exp_id": "E03", "branch": "proposed-method", "model": "Dual-Branch Fusion", "lr": 1e-4, "epochs": 25, "val_acc": 0.00, "macro_f1": 0.00},
]

if experiments_log:
    exp_df = pd.DataFrame(experiments_log)
    exp_df.to_csv(RESULTS_DIR / "experiments.csv", index=False)
    display(exp_df)
else:
    print("No experiments logged yet. Add runs to experiments_log as you train.")

No experiments logged yet. Add runs to experiments_log as you train.


Save your work, then run these commands in the terminal:
```bash
git status
git add .
git commit -m "EXP: E01 baseline, accuracy 0.81"
git push
```

# Testing and validation
*FPR: Methodology (test strategy and validation)*

- **Testing:** Verify data shapes, absence of missing values, valid probability outputs, and zero train/test overlap.
- **Validation:** Evaluate reliability using stratified k-fold cross-validation and an unseen held-out test set.

In [8]:
# Unit tests for pipeline integrity (uncomment and adapt as functions are implemented)
def run_unit_tests():
    # assert clean_metadata(df).isna().sum().sum() == 0, "Cleaned data must contain no NaNs"
    # assert len(set(train_ids).intersection(set(test_ids))) == 0, "Data leakage detected between train and test"
    # assert probs.shape == (len(X_test), num_classes), "Unexpected prediction shape"
    print("All pipeline unit tests passed.")

run_unit_tests()

All pipeline unit tests passed.


In [9]:
# validation, e.g. k-fold cross-validation or a held-out test set

Save your work, then run these commands in the terminal:
```bash
git status
git add .
git commit -m "TEST: unit tests and 5-fold cross-validation"
git push
```

# Results
*FPR: Quality and Results*

Save every figure and table you use in the report, then upload them to Overleaf.

In [10]:
# Load logged experiments, display summary, and export LaTeX table for Overleaf
results_file = RESULTS_DIR / "experiments.csv"
if results_file.exists():
    results = pd.read_csv(results_file)
    display(results)
    results.to_latex(FIG_DIR / "results_table.tex", index=False)
    print("Exported results_table.tex to figures/ for Overleaf.")
else:
    print("Run experiments and save results/experiments.csv first.")# results.to_latex(FIG_DIR / "results_table.tex", index=False)   # paste or \input into Overleaf

Run experiments and save results/experiments.csv first.


*What do the results mean? Link them to your objectives and the literature.*

Save your work, then run these commands in the terminal:
```bash
git status
git add .
git commit -m "RESULT: final results table and figures for FPR"
git push
```

# Demo / viva preparation (20%)
About 10 minutes of demo, then about 10 minutes of questions. Focus on the practical work: no slides, no reading the report.

**Plan what to show:**
1. Problem and objective (1 min)
2. Data and how you prepared it
3. Your artefact running: run the cell below from a fresh kernel (`Kernel -> Restart and Run All`)
4. Key results and what they mean

**Practise these questions:**
- What did you do, why, and how? How did you know how to do it?
- What challenges did you meet, and how did you solve them?
- How did your project change over time? (use your commit history)
- Explain this function and change a parameter live. What happens?
- Where could your code be optimised? Show the change.

In [11]:
# DEMO: run the whole pipeline end-to-end from a fresh kernel (Kernel -> Restart and Run All)
def run_full_demo():
    print("1. Loading and verifying dataset...")
    # data = load_data(...)
    print("2. Running preprocessing and unit checks...")
    run_unit_tests()
    print("3. Loading trained fusion model and running inference on sample test batch...")
    # preds = predict_sample(...)
    print("4. Displaying evaluation summary and Grad-CAM visualisation...")
    # plot_demo_results(...)
    print("Demo pipeline completed successfully.")

run_full_demo()

1. Loading and verifying dataset...
2. Running preprocessing and unit checks...
All pipeline unit tests passed.
3. Loading trained fusion model and running inference on sample test batch...
4. Displaying evaluation summary and Grad-CAM visualisation...
Demo pipeline completed successfully.


# Supervision log

| Date | Discussion summary | Agreed actions before next meeting |
| :--- | :--- | :--- |
| YYYY-MM-DD | Initial project scope, dataset selection, and VIP GitHub/Overleaf setup | Finalise preprocessing pipeline on `baseline` branch and log initial EDA figures |

---

# Create your release (before you submit the IPR and FPR)

| Milestone | Release name |
| :--- | :--- |
| Interim Progress Report (10%) | `IPR-v1.0` |
| Final Project Report (70%) | `FPR-v1.0` |
| Optional fixes before demo | `Demo-v1.0` |

1. **Commit and push your final work:**
   ```bash
   git status
   git add .
   git commit -m "FPR: final version for submission"
   git push